# Model Comparison & Ablation Study

This notebook compares the performance of different fraud detection models:
1.  **Baseline XGBoost** (Tabular features only)
2.  **Hybrid Models** (XGBoost + GNN Embeddings)
    *   GAT + XGBoost
    *   GCN + XGBoost
    *   HGT + XGBoost
    *   HGT-RTE + XGBoost

## Metrics
*   **AUC-PR**: Area Under the Precision-Recall Curve (Primary Metric)
*   **Precision@100**: Precision of the top 100 riskiest listings (Operational Metric)

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
def load_results():
    results = {}
    path = "../artifacts/results/*.csv"
    for file in glob.glob(path):
        name = os.path.basename(file).replace("_results.csv", "")
        df = pl.read_csv(file)
        # Convert window_start to datetime
        df = df.with_columns(pl.col("window_start").str.to_date())
        results[name] = df
    return results

results = load_results()
print(f"Loaded models: {list(results.keys())}")

## AUC-PR Over Time

In [ ]:
plt.figure(figsize=(14, 7))

for name, df in results.items():
    # Sort by date
    df = df.sort("window_start")
    dates = df["window_start"].to_list()
    scores = df["auc_pr"].to_list()
    
    plt.plot(dates, scores, marker='o', label=name, linewidth=2)

plt.title("Model Performance (AUC-PR) Over Time", fontsize=16)
plt.xlabel("Window Start Date", fontsize=12)
plt.ylabel("AUC-PR", fontsize=12)
plt.legend(title="Model", fontsize=10)
plt.xticks(rotation=45)
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
summary = []
for name, df in results.items():
    mean_auc = df["auc_pr"].mean()
    # Check if p@100 exists (baseline might not have it if older version, but new one does)
    if "p@100" in df.columns:
        mean_p100 = df["p@100"].mean()
    else:
        mean_p100 = None
        
    summary.append({
        "Model": name,
        "Mean AUC-PR": mean_auc,
        "Mean P@100": mean_p100,
        "Windows Evaluated": len(df)
    })

summary_df = pl.DataFrame(summary).sort("Mean AUC-PR", descending=True)
print(summary_df)